# 🤖 Notebook 03 — Text Preprocessing & Sentiment Analysis
## Early Warning System Krisis Pariwisata Bali

Notebook ini memproses:
1. Bali Hotel Review (semicolon-separated, multi-kolom)
2. Dataset of Digital Reviews in Tourism (hanya kolom review)
3. Text cleaning (lowercase, hapus URL, angka, tanda baca)
4. Deteksi bahasa (langdetect)
5. Sentiment analysis dengan Transformer (XLM-RoBERTa multilingual)
6. Agregasi sentimen bulanan

**Output:** `data/processed/monthly_sentiment.csv`

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print('✅ Library dasar siap')

✅ Library dasar siap


## 1. Load merged_all_hotels.xlsx

Dataset ini menggabungkan semua review hotel Bali dengan kolom:
`date` | `location` | `hotel` | `review` | `rating`

- **29.332 baris**, rentang **2009–2026**
- Kolom `date` sudah bertipe `datetime64` — tidak perlu konversi serial Excel

In [2]:
# ─── Load merged_all_hotels.xlsx ──────────────────────────────────────────
# Kolom: date | location | hotel | review | rating
# date sudah datetime64 (openpyxl auto-parse)

reviews = pd.read_excel(
    'data/raw/merged_all_hotels.xlsx',
    parse_dates=['date']
)

# Normalisasi kolom rating ke numerik
reviews['rating'] = pd.to_numeric(reviews['rating'], errors='coerce')

print('Shape            :', reviews.shape)
print('Kolom            :', reviews.columns.tolist())
print('Tipe date        :', reviews['date'].dtype)
print('Rentang tanggal  :', reviews['date'].min(), '→', reviews['date'].max())
print('Null rating      :', reviews['rating'].isna().sum(), '/', len(reviews))
print()
reviews.head(3)

Shape            : (29332, 5)
Kolom            : ['date', 'location', 'hotel', 'review', 'rating']
Tipe date        : object
Rentang tanggal  : 2009-05-03 23:59:59.999000 → 2026-05-25 05:00:10.629000
Null rating      : 3704 / 29332



,date,location,hotel,review,rating
0,2009-05-03 23:59:59.999000,"Jl. Segara Ayu, Sanur, Denpasar Selatan, Kota ...",Segara Village Hotel,Мне не понравился Санур и данный отель. Первый...,NaN
1,2009-09-01 23:59:59.999000,"Jl. Segara Ayu, Sanur, Denpasar Selatan, Kota ...",Segara Village Hotel,"Las habitaciones son muy modernas,aunque para ...",NaN
2,2009-10-12 23:59:59.999000,"Jl. Segara Ayu, Sanur, Denpasar Selatan, Kota ...",Segara Village Hotel,Segara Village酒店的位置不错，非常安静。酒店内部是地道的巴厘岛风情园林。在酒店...,NaN


In [3]:
# Preview kolom yang akan dipakai di pipeline
print('Kolom review — sample:')
print(reviews['review'].head(2).to_string())
print()
print('Distribusi rating:')
print(reviews['rating'].value_counts().sort_index().to_string())

Kolom review — sample:
0    Мне не понравился Санур и данный отель. Первый...
1    Las habitaciones son muy modernas,aunque para ...

Distribusi rating:
rating
1.0      857
2.0      540
3.0     1146
4.0     3128
5.0    19957


---

## 2. Load Dataset of Digital Reviews in Tourism

In [4]:
# File ini hanya punya 1 kolom: 'review'
digital_reviews = pd.read_csv(
    'data/raw/Dataset of Digital Reviews in Tourism - 2.csv',
    engine='python',
    on_bad_lines='skip'
)

print('Shape:', digital_reviews.shape)
print('Columns:', digital_reviews.columns.tolist())

# Rename jika perlu
if 'review' not in digital_reviews.columns:
    digital_reviews.columns = ['review']

digital_reviews.head(3)

Shape: (56448, 1)
Columns: ['review']


,review
0,An out of this world experience! Gita was an a...
1,What an amazing experience. Not only Orang-uta...
2,Good experience with the friendly staff at the...


## 3. Gabung Semua Dataset Review

In [5]:
# Gabung merged_hotels + digital_reviews menjadi 1 dataframe
# Kolom 'date' dibawa dari merged_hotels; digital_reviews tidak punya tanggal → NaT

reviews['source'] = 'merged_hotels'

digital_reviews_copy = digital_reviews[['review']].copy()
digital_reviews_copy['source'] = 'digital_tourism'
digital_reviews_copy['date']   = pd.NaT          # tidak ada tanggal
digital_reviews_copy['rating'] = np.nan           # tidak ada rating

all_reviews = pd.concat([reviews, digital_reviews_copy], ignore_index=True)

print('Total reviews gabungan :', len(all_reviews))
print('Per sumber             :')
print(all_reviews['source'].value_counts().to_string())
print()
print('Baris dengan date      :', all_reviews['date'].notna().sum())
print('Baris tanpa date       :', all_reviews['date'].isna().sum())
all_reviews.head(3)

Total reviews gabungan : 85780
Per sumber             :
source
digital_tourism    56448
merged_hotels      29332

Baris dengan date      : 29332
Baris tanpa date       : 56448


,date,location,hotel,review,rating,source
0,2009-05-03 23:59:59.999000,"Jl. Segara Ayu, Sanur, Denpasar Selatan, Kota ...",Segara Village Hotel,Мне не понравился Санур и данный отель. Первый...,NaN,merged_hotels
1,2009-09-01 23:59:59.999000,"Jl. Segara Ayu, Sanur, Denpasar Selatan, Kota ...",Segara Village Hotel,"Las habitaciones son muy modernas,aunque para ...",NaN,merged_hotels
2,2009-10-12 23:59:59.999000,"Jl. Segara Ayu, Sanur, Denpasar Selatan, Kota ...",Segara Village Hotel,Segara Village酒店的位置不错，非常安静。酒店内部是地道的巴厘岛风情园林。在酒店...,NaN,merged_hotels


## 4. Text Cleaning

In [6]:
def clean_text(text):
    """
    Fungsi pembersih teks untuk review wisatawan.
    Kompatibel dengan teks multibahasa (EN, ID, ZH).
    """
    if pd.isna(text) or text == '':
        return ''
    
    text = str(text)
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Hapus URL
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # 3. Hapus HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # 4. Hapus karakter khusus tapi pertahankan huruf, angka, spasi
    # Catatan: untuk teks Mandarin, kita pertahankan karakter unicode
    text = re.sub(r'[^\w\s\u4e00-\u9fff]', ' ', text)
    
    # 5. Hapus angka yang berdiri sendiri
    text = re.sub(r'\b\d+\b', '', text)
    
    # 6. Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Terapkan cleaning
all_reviews['review_clean'] = all_reviews['review'].apply(clean_text)

# Hapus review kosong
all_reviews = all_reviews[all_reviews['review_clean'].str.len() > 10].copy()

# Hapus duplikat
all_reviews = all_reviews.drop_duplicates(subset=['review_clean']).reset_index(drop=True)

print('Setelah cleaning - Total reviews:', len(all_reviews))
print()
print('Contoh hasil cleaning:')
for i in range(3):
    print(f'  Original : {str(all_reviews["review"].iloc[i])[:80]}')
    print(f'  Cleaned  : {str(all_reviews["review_clean"].iloc[i])[:80]}')
    print()

Setelah cleaning - Total reviews: 82236

Contoh hasil cleaning:
  Original : Мне не понравился Санур и данный отель. Первый номер, который я получила при зас
  Cleaned  : мне не понравился санур и данный отель первый номер который я получила при засел

  Original : Las habitaciones son muy modernas,aunque para nuestro gusto un poco pequeñas. Ti
  Cleaned  : las habitaciones son muy modernas aunque para nuestro gusto un poco pequeñas tie

  Original : Segara Village酒店的位置不错，非常安静。酒店内部是地道的巴厘岛风情园林。在酒店的中心处有两个巨大的泳池，非常干净。酒店的沙滩上晒晒太阳，在酒店的各
  Cleaned  : segara village酒店的位置不错 非常安静 酒店内部是地道的巴厘岛风情园林 在酒店的中心处有两个巨大的泳池 非常干净 酒店的沙滩上晒晒太阳 在酒店的各



## 5. Deteksi Bahasa

In [7]:
from langdetect import detect, LangDetectException

def detect_language(text):
    """Deteksi bahasa teks. Return 'unknown' jika gagal."""
    try:
        if len(str(text).split()) < 3:
            return 'unknown'
        return detect(str(text))
    except LangDetectException:
        return 'unknown'

# Deteksi bahasa (ini mungkin butuh beberapa menit untuk dataset besar)
print('Mendeteksi bahasa... (mungkin butuh beberapa menit)')
all_reviews['language'] = all_reviews['review_clean'].apply(detect_language)

# Ringkasan distribusi bahasa
lang_dist = all_reviews['language'].value_counts().head(10)
print('\nDistribusi Bahasa:')
print(lang_dist.to_string())

# Filter bahasa yang relevan: EN, ID, ZH (sesuai proyek)
target_languages = ['en', 'id', 'zh-cn', 'zh-tw', 'ms']
print(f'\nReviews dengan bahasa target {target_languages}: {all_reviews[all_reviews["language"].isin(target_languages)].shape[0]}')

Mendeteksi bahasa... (mungkin butuh beberapa menit)

Distribusi Bahasa:
language
en         73295
id          3190
de           923
ko           894
fr           811
unknown      474
es           402
nl           388
ru           331
it           242

Reviews dengan bahasa target ['en', 'id', 'zh-cn', 'zh-tw', 'ms']: 76618


## 6. Sentiment Analysis dengan Transformer

Menggunakan **XLM-RoBERTa** yang mendukung analisis sentimen multibahasa (EN, ID, ZH).

⚠️ **Model ini akan download sekitar 500MB pertama kali dijalankan.**
Pastikan koneksi internet aktif.

In [8]:
from transformers import pipeline
import torch

# Cek apakah GPU tersedia
device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU" if device == 0 else "CPU"}')

# Load model XLM-RoBERTa untuk sentimen multibahasa
# Model ini support: EN, ID, ZH, dan 100+ bahasa lainnya
print('Loading model sentiment analysis...')
sentiment_analyzer = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
    device=device,
    max_length=512,
    truncation=True
)
print('✅ Model berhasil dimuat!')

Device: GPU
Loading model sentiment analysis...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model berhasil dimuat!


In [9]:
# Test model dengan contoh teks
test_texts = [
    'This hotel is amazing! Great service and beautiful view.',  # EN positif
    'Pelayanannya sangat buruk dan kamarnya kotor sekali.',       # ID negatif
    'Great location but the food was just okay.',                  # EN netral
]

print('Test Sentiment Analysis:')
for text in test_texts:
    result = sentiment_analyzer(text[:512])[0]
    print(f'  Teks  : {text[:60]}')
    print(f'  Hasil : {result["label"]} (score: {result["score"]:.3f})')
    print()

Test Sentiment Analysis:
  Teks  : This hotel is amazing! Great service and beautiful view.
  Hasil : positive (score: 0.942)

  Teks  : Pelayanannya sangat buruk dan kamarnya kotor sekali.
  Hasil : negative (score: 0.950)

  Teks  : Great location but the food was just okay.
  Hasil : positive (score: 0.843)



In [10]:
def get_sentiment_score(text):
    """
    Ambil skor sentimen dari teks.
    Return: float antara -1 (sangat negatif) sampai 1 (sangat positif)
    """
    try:
        text_clean = str(text)[:512]  # limit token
        if len(text_clean.strip()) < 5:
            return 0.0
        
        result = sentiment_analyzer(text_clean)[0]
        label = result['label'].lower()
        score = result['score']
        
        # Konversi ke skala -1 sampai 1
        if 'positive' in label or label == 'pos':
            return score          # positif
        elif 'negative' in label or label == 'neg':
            return -score         # negatif
        else:
            return 0.0            # netral
    except Exception:
        return 0.0

def get_sentiment_label(score):
    """Konversi skor ke label."""
    if score > 0.1:
        return 'positive'
    elif score < -0.1:
        return 'negative'
    else:
        return 'neutral'

print('Fungsi sentiment siap!')
print('Contoh:')
print('  Score:', get_sentiment_score('This hotel is terrible and the staff is rude'))

def get_sentiment_confidence(text):
    """
    Ambil confidence score dari model sentiment.
    Return: float antara 0-1 (seberapa yakin model dengan prediksinya)
    """
    try:
        text_clean = str(text)[:512]
        if len(text_clean.strip()) < 5:
            return 0.0
        result = sentiment_analyzer(text_clean)[0]
        return float(result['score'])   # probability dari label yang dipilih
    except Exception:
        return 0.0


Fungsi sentiment siap!
Contoh:
  Score: -0.961677074432373


In [11]:
# ✅ Proses SEMUA reviews dengan batch processing (jauh lebih cepat di GPU)
print(f'Total reviews: {len(all_reviews)} baris')

reviews_sample = all_reviews.copy()

print(f'Memproses {len(reviews_sample)} reviews...')

# Batch processing — lebih cepat dibanding .apply()
BATCH_SIZE = 32

# Ambil semua teks sebagai list
texts = reviews_sample["review_clean"].fillna("").astype(str).tolist()

# Minimal panjang teks
texts = [t[:512] if len(t) >= 5 else "neutral" for t in texts]

# Jalankan batch inference
print(f"Menjalankan batch inference (batch_size={BATCH_SIZE})...")

results = sentiment_analyzer(
    texts,
    batch_size=BATCH_SIZE,
    truncation=True,
    max_length=512
)

# Konversi hasil ke skor -1 sampai 1
def result_to_score(r):
    label = r["label"].lower()
    score = r["score"]

    if "positive" in label or label == "pos":
        return score

    elif "negative" in label or label == "neg":
        return -score

    else:
        return 0.0


reviews_sample["sentiment_score"] = [
    result_to_score(r) for r in results
]

reviews_sample["sentiment_label"] = reviews_sample[
    "sentiment_score"
].apply(get_sentiment_label)

print("\n✅ Sentiment analysis selesai!")

print("\nDistribusi sentimen:")
print(
    reviews_sample["sentiment_label"]
    .value_counts()
    .to_string()
)

print(
    f"\nRata-rata skor: "
    f"{reviews_sample['sentiment_score'].mean():.3f}"
)

print(f"Total diproses: {len(reviews_sample)} reviews")

Total reviews: 82236 baris
Memproses 82236 reviews...
Menjalankan batch inference (batch_size=32)...


KeyboardInterrupt: 

In [ ]:
# Simpan hasil sentiment
reviews_sample.to_csv('data/processed/reviews_with_sentiment.csv', index=False)
print('✅ reviews_with_sentiment.csv disimpan')

reviews_sample[['review_clean', 'language', 'sentiment_score', 'sentiment_label']].head(5)

In [ ]:
# # ✅ TAMBAHAN: Hitung confidence score untuk setiap review
# # Confidence = seberapa yakin model XLM-R dengan prediksi sentimen-nya
# print('Menghitung confidence score...')

# # Gunakan sample agar tidak terlalu lama (confidence membutuhkan 1 forward pass lagi)
# conf_sample = reviews_sample.copy()
# conf_sample['sentiment_confidence'] = conf_sample['review'].apply(get_sentiment_confidence)

# # Gabungkan ke reviews_sample
# reviews_sample['sentiment_confidence'] = conf_sample['sentiment_confidence']

# print(f'✅ Confidence ditambahkan ke reviews_sample')
# print()
# print('=== DISTRIBUSI CONFIDENCE ===')
# print(conf_sample['sentiment_confidence'].describe().round(3).to_string())
# print()

# # Distribusi: berapa % review punya confidence tinggi (>0.8)
# high_conf = (conf_sample['sentiment_confidence'] >= 0.8).mean() * 100
# low_conf  = (conf_sample['sentiment_confidence'] < 0.5).mean() * 100
# print(f'Confidence tinggi (≥0.8) : {high_conf:.1f}%')
# print(f'Confidence rendah (<0.5) : {low_conf:.1f}%')
# print()

# # Cross-tabulation: confidence vs sentiment_label
# conf_by_label = conf_sample.groupby('sentiment_label')['sentiment_confidence'].agg(['mean','min','max'])
# print('=== CONFIDENCE PER LABEL ===')
# print(conf_by_label.round(3).to_string())


In [ ]:
# ✅ TAMBAHAN: Hitung confidence score untuk setiap review
# Confidence = seberapa yakin model XLM-R dengan prediksi sentimen-nya

print('Menghitung confidence score...')

# Gunakan sample agar tidak terlalu lama
conf_sample = reviews_sample.copy()

# Hitung confidence
conf_sample['sentiment_confidence'] = (
    conf_sample['review']
    .fillna('')
    .astype(str)
    .apply(get_sentiment_confidence)
)

# Gabungkan kembali ke reviews_sample
reviews_sample['sentiment_confidence'] = conf_sample['sentiment_confidence']

# ============================================================
# SIMPAN HASIL SENTIMENT KE ALL_REVIEWS
# ============================================================

all_reviews['sentiment_score'] = reviews_sample['sentiment_score'].values
all_reviews['sentiment_label'] = reviews_sample['sentiment_label'].values
all_reviews['sentiment_confidence'] = reviews_sample['sentiment_confidence'].values

print('✅ Kolom sentiment berhasil digabung ke all_reviews')
print()

# ============================================================
# CEK DISTRIBUSI LABEL
# ============================================================

print('=== DISTRIBUSI LABEL SENTIMENT ===')
print(
    (
        all_reviews['sentiment_label']
        .value_counts(normalize=True) * 100
    ).round(2).to_string()
)

print()

# ============================================================
# STATISTIK CONFIDENCE
# ============================================================

print('=== DISTRIBUSI CONFIDENCE ===')
print(
    conf_sample['sentiment_confidence']
    .describe()
    .round(3)
    .to_string()
)

print()

# Distribusi confidence tinggi/rendah
high_conf = (
    all_reviews['sentiment_confidence'] >= 0.8
).mean() * 100

low_conf = (
    all_reviews['sentiment_confidence'] < 0.5
).mean() * 100

print(f'Confidence tinggi (≥0.8) : {high_conf:.1f}%')
print(f'Confidence rendah (<0.5) : {low_conf:.1f}%')

print()

# ============================================================
# CONFIDENCE PER LABEL
# ============================================================

conf_by_label = (
    all_reviews
    .groupby('sentiment_label')['sentiment_confidence']
    .agg(['mean','min','max'])
)

print('=== CONFIDENCE PER LABEL ===')
print(conf_by_label.round(3).to_string())

print()

# ============================================================
# SIMPAN CSV FINAL
# ============================================================

import os

os.makedirs('data/processed', exist_ok=True)

all_reviews.to_csv(
    'data/processed/all_reviews_sentiment.csv',
    index=False
)

print('✅ File berhasil disimpan:')
print('data/processed/all_reviews_sentiment.csv')

print()
print('Kolom final:')
print(all_reviews.columns.tolist())

In [ ]:
print(all_reviews["sentiment_label"].value_counts(normalize=True) * 100)

## 7. Sentimen Berdasarkan Rating (Jika Ada)

Untuk Bali_Hotel_Review.csv yang punya kolom rating, kita bisa validasi sentimen transformer dengan rating aktual.

In [ ]:
if 'rating' in reviews_sample.columns:
    # Analisis korelasi rating vs sentiment score
    hotel_only = reviews_sample[reviews_sample['source'] == 'bali_hotel'].dropna(subset=['rating'])
    
    if len(hotel_only) > 0:
        corr = hotel_only['rating'].corr(hotel_only['sentiment_score'])
        print(f'Korelasi Rating vs Sentiment Score: {corr:.3f}')
        
        # Distribusi sentimen per rating
        print('\nRata-rata sentiment score per rating:')
        print(hotel_only.groupby('rating')['sentiment_score'].mean().round(3).to_string())
else:
    print('Kolom rating tidak tersedia di dataset ini.')

## 8. Agregasi Sentimen Bulanan

✅ **Dataset `merged_all_hotels.xlsx` memiliki kolom tanggal** — sentimen sekarang bisa diagregasi per bulan!
Ini menyelesaikan masalah *static sentiment* yang sebelumnya membuat `crisis_component_sentiment` bernilai konstan.

### 8a. Defensive Date Parse

Meskipun `merged_all_hotels.xlsx` sudah menghasilkan `datetime64`, fungsi ini
dipasang sebagai **safeguard** — jika suatu saat file CSV/Excel lain dipakai dan
kolom `date` terbaca sebagai *Excel serial number* (misal `39937.0`),
konversi tetap benar.

In [ ]:
# ─── Defensive: konversi date yang mungkin berupa Excel serial number ──────
# merged_all_hotels sudah datetime64, tapi ini safeguard untuk dataset lain.

def parse_date_safe(val):
    """
    Handle dua kasus:
    1. Sudah datetime / string  → langsung parse
    2. Excel serial number      → konversi dari epoch 1899-12-30
    """
    if pd.isna(val):
        return pd.NaT
    # Sudah Timestamp → kembalikan langsung
    if isinstance(val, pd.Timestamp):
        return val
    try:
        numeric_val = float(val)
        if numeric_val >= 1000:   # serial number (1 Jan 1900 = 1)
            return pd.Timestamp('1899-12-30') + pd.Timedelta(days=numeric_val)
        return pd.NaT
    except (ValueError, TypeError):
        try:
            return pd.to_datetime(val)
        except Exception:
            return pd.NaT

# Terapkan ke reviews_sample sebelum agregasi bulanan
if 'date' in reviews_sample.columns:
    reviews_sample['date'] = reviews_sample['date'].apply(parse_date_safe)

    valid  = reviews_sample['date'].notna().sum()
    total  = len(reviews_sample)
    sample = reviews_sample[reviews_sample['date'].notna()][['date']].head(3)

    print(f'✅ Date valid : {valid}/{total} baris')
    print(f'   Rentang   : {reviews_sample["date"].min()} → {reviews_sample["date"].max()}')
    print()
    print('Contoh tanggal setelah parse:')
    print(sample.to_string())
else:
    print('⚠️  Kolom date tidak ditemukan di reviews_sample')

In [ ]:
# ✅ Monthly Sentiment Aggregation — menggunakan tanggal asli merged_all_hotels
# date sudah Timestamp (setelah parse_date_safe di atas)

reviews_with_date = reviews_sample[reviews_sample['date'].notna()].copy()
reviews_with_date['month'] = reviews_with_date['date'].dt.to_period('M')

# Agregasi per bulan
monthly_sentiment = reviews_with_date.groupby('month').agg(
    avg_sentiment = ('sentiment_score', 'mean'),
    pct_positive  = ('sentiment_label', lambda x: (x == 'positive').mean() * 100),
    pct_negative  = ('sentiment_label', lambda x: (x == 'negative').mean() * 100),
    pct_neutral   = ('sentiment_label', lambda x: (x == 'neutral').mean() * 100),
    review_count  = ('sentiment_score', 'count'),
    avg_rating    = ('rating', 'mean')
).reset_index()

monthly_sentiment['month'] = monthly_sentiment['month'].astype(str)

print('=== Monthly Sentiment Coverage ===')
print(f'Bulan dengan data : {len(monthly_sentiment)}')
print(f'Rentang           : {monthly_sentiment["month"].min()} → {monthly_sentiment["month"].max()}')
print()
print('Sample 5 bulan terakhir:')
print(monthly_sentiment.tail(5).round(3).to_string())
print()

# Statistik global — fallback untuk bulan tanpa data di NB04
overall_sentiment = {
    'avg_sentiment' : reviews_sample['sentiment_score'].mean(),
    'pct_positive'  : (reviews_sample['sentiment_label'] == 'positive').mean() * 100,
    'pct_negative'  : (reviews_sample['sentiment_label'] == 'negative').mean() * 100,
    'pct_neutral'   : (reviews_sample['sentiment_label'] == 'neutral').mean() * 100,
    'total_reviews' : len(reviews_sample)
}
print('=== Statistik Global (fallback NB04) ===')
for k, v in overall_sentiment.items():
    print(f'  {k}: {v:.2f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# ✅ TAMBAHAN: Review volume sebagai sinyal tambahan
# Penurunan jumlah review = sinyal penurunan minat wisatawan (kurang data = bahaya)

print('=== REVIEW VOLUME ANALYSIS ===')
print()
print(f'Total bulan dengan data : {len(monthly_sentiment)}')
print(f'Rata-rata review/bulan  : {monthly_sentiment["review_count"].mean():.1f}')
print(f'Min review/bulan        : {monthly_sentiment["review_count"].min()}')
print(f'Max review/bulan        : {monthly_sentiment["review_count"].max()}')
print()

# Flag bulan dengan volume review sangat rendah (< 10 review = tidak representatif)
low_volume = monthly_sentiment[monthly_sentiment['review_count'] < 10]
print(f'Bulan dengan review < 10: {len(low_volume)}')
if len(low_volume) > 0:
    print('  → Bulan-bulan ini sentimen-nya kurang reliable:')
    print(low_volume[['month','review_count','avg_sentiment']].to_string())

# Tambah kolom: apakah volume review cukup untuk dijadikan sinyal?
monthly_sentiment['is_volume_reliable'] = (monthly_sentiment['review_count'] >= 10).astype(int)

# Tambah ke file yang disimpan
monthly_sentiment.to_csv('data/processed/monthly_sentiment.csv', index=False)
print()
print('✅ monthly_sentiment.csv diperbarui dengan kolom is_volume_reliable')


In [ ]:
# Simpan monthly_sentiment.csv — dipakai di NB04 sebagai fitur time-series
monthly_sentiment.to_csv('data/processed/monthly_sentiment.csv', index=False)

# Simpan juga sentiment_stats.csv (nilai global sebagai fallback)
sentiment_stats = pd.DataFrame([overall_sentiment])
sentiment_stats.to_csv('data/processed/sentiment_stats.csv', index=False)

# Simpan semua reviews dengan sentiment
save_cols = ['review_clean', 'language', 'sentiment_score', 'sentiment_label', 'source']
if 'date' in reviews_sample.columns:
    save_cols = ['date'] + save_cols
reviews_sample[save_cols].to_csv('data/processed/all_reviews_sentiment.csv', index=False)

print('✅ monthly_sentiment.csv disimpan  ← BARU: sentimen per bulan (bervariasi!)')
print('✅ sentiment_stats.csv disimpan    ← fallback: sentimen global')
print('✅ all_reviews_sentiment.csv disimpan')
print()
print('Kolom monthly_sentiment:')
print(monthly_sentiment.columns.tolist())
print()
print('Langkah selanjutnya:')
print('  → NB04 akan merge monthly_sentiment berdasarkan kolom month')
print('  → crisis_component_sentiment sekarang bervariasi per bulan!')

## 9. Analisis Sentimen per Bahasa

In [ ]:
# Analisis sentimen berdasarkan bahasa
lang_sentiment = reviews_sample.groupby('language').agg(
    total=('sentiment_score', 'count'),
    avg_score=('sentiment_score', 'mean'),
    pct_positive=('sentiment_label', lambda x: (x == 'positive').mean() * 100),
    pct_negative=('sentiment_label', lambda x: (x == 'negative').mean() * 100)
).round(2)

# Sort berdasarkan total terbanyak
lang_sentiment = lang_sentiment.sort_values('total', ascending=False)

print('=== Analisis Sentimen per Bahasa ===')
print(lang_sentiment.head(10).to_string())

In [ ]:
import matplotlib.pyplot as plt

# Visualisasi distribusi sentimen
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart distribusi label
label_counts = reviews_sample['sentiment_label'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#95a5a6']
axes[0].pie(label_counts.values, labels=label_counts.index, colors=colors, autopct='%1.1f%%')
axes[0].set_title('Distribusi Sentimen Review Wisatawan')

# Histogram skor sentimen
axes[1].hist(reviews_sample['sentiment_score'], bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', label='Netral (0)')
axes[1].set_title('Distribusi Skor Sentimen (-1 hingga +1)')
axes[1].set_xlabel('Sentiment Score')
axes[1].set_ylabel('Jumlah Review')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/processed/sentiment_distribution.png', dpi=150)
plt.show()
print('✅ Plot disimpan ke data/processed/sentiment_distribution.png')